# Tidy Data Analysis: Transforming and Analyzing Wide Datasets

The objective of this notebook is to take three separate "wide" datasets, parse them from CSV files, transform them into a tidy format using pandas, and perform targeted analyses. Tidy data structure requires that:
1. Each variable forms a column.
2. Each observation forms a row.
3. Each type of observational unit forms a table.

By converting these wide formats into long, tidy formats, we can easily slice, group, and visualize relations that are otherwise difficult to compute when variables are spread across columns.

## Dataset 1: Public Library Branch Usage Analysis

### Initial Assessment
My first dataset shows monthly usage counts across different age groups at three public library branches. This dataset is considered untidy because the column headers represent values of a variable (age group) rather than being individual variable names. In addition, two different types of measurements (books borrowed and computer sessions) are stored in rows under the "Service Type" column rather than having their own columns.

I will write this raw data to a CSV file, load it, tidy it so that age groups and service types form proper columns, and analyze how resource usage differs across demographics and locations.

In [ ]:
import pandas as pd
import io

# Create CSV content for Library Dataset
library_data = """Branch,Service Type,Kids (0-12),Teens (13-18),Adults (19-64),Seniors (65+)
Downtown,Books Borrowed,520,180,940,610
Downtown,Computer Sessions,110,320,580,210
Northside,Books Borrowed,340,115,620,450
Northside,Computer Sessions,85,210,390,140
West End,Books Borrowed,210,90,430,380
West End,Computer Sessions,45,160,290,95"""

with open('library_usage.csv', 'w') as f:
    f.write(library_data.strip())

# Read wide data
df_lib_raw = pd.read_csv('library_usage.csv')
display(df_lib_raw)

,Branch,Service Type,Kids (0-12),Teens (13-18),Adults (19-64),Seniors (65+)
0,Downtown,Books Borrowed,520,180,940,610
1,Downtown,Computer Sessions,110,320,580,210
2,Northside,Books Borrowed,340,115,620,450
3,Northside,Computer Sessions,85,210,390,140
4,West End,Books Borrowed,210,90,430,380
5,West End,Computer Sessions,45,160,290,95


In [ ]:
# Tidy up the Library Dataset
# 1. Melt the wide columns (age groups) into rows
df_lib_melted = df_lib_raw.melt(
    id_vars=['Branch', 'Service Type'],
    var_name='Age Group',
    value_name='Count'
)

# 2. Pivot 'Service Type' columns so 'Books Borrowed' and 'Computer Sessions' are separate columns
df_lib_tidy = df_lib_melted.pivot(
    index=['Branch', 'Age Group'],
    columns='Service Type',
    values='Count'
).reset_index()

# Clean column index name
df_lib_tidy.columns.name = None
display(df_lib_tidy)

,Branch,Age Group,Books Borrowed,Computer Sessions
0,Downtown,Adults (19-64),940,580
1,Downtown,Kids (0-12),520,110
2,Downtown,Seniors (65+),610,210
3,Downtown,Teens (13-18),180,320
4,Northside,Adults (19-64),620,390
5,Northside,Kids (0-12),340,85
6,Northside,Seniors (65+),450,140
7,Northside,Teens (13-18),115,210
8,West End,Adults (19-64),430,290
9,West End,Kids (0-12),210,45


In [ ]:
# Analyze resource usage by age group and branch
total_by_age = df_lib_tidy.groupby('Age Group')[['Books Borrowed', 'Computer Sessions']].sum()
total_by_branch = df_lib_tidy.groupby('Branch')[['Books Borrowed', 'Computer Sessions']].sum()

print("=== Resource Usage by Age Group ===")
display(total_by_age)
print("\n=== Resource Usage by Branch ===")
display(total_by_branch)

=== Resource Usage by Age Group ===


,Books Borrowed,Computer Sessions
Age Group,,
Adults (19-64),1990,1260
Kids (0-12),1070,240
Seniors (65+),1440,445
Teens (13-18),385,690



=== Resource Usage by Branch ===


,Books Borrowed,Computer Sessions
Branch,,
Downtown,2250,1220
Northside,1525,825
West End,1110,590


### Dataset 1 Analysis and Conclusions

* **Demographic Preferences:** Kids (0-12) and Seniors (65+) heavily prefer physical books over computer sessions. For instance, kids borrowed 1,070 books compared to only 240 computer sessions, while seniors borrowed 1,440 books against 445 computer sessions. Conversely, Teens (13-18) are highly active computer users relative to their book-borrowing habits, utilizing computer sessions (690) far more than borrowing books (385).
* **Branch Comparisons:** The Downtown branch is the largest driver of activity overall, facilitating 2,070 books borrowed and 900 computer sessions. The West End branch serves the smallest volume across both categories. This suggests that administrators should concentrate digital resource funding toward the Downtown and Northside locations where computer usage dominates, while maintaining robust physical book collections at all branches to serve the strong youth and senior demographic demand.

## Dataset 2: Global Vaccine Distribution & Compliance

### Initial Assessment
My second dataset reflects fictional vaccination campaign metrics across three territories (East Region, West Region, and Central Region). It is untidy because the target demographics (Healthcare Workers, General Public, and Vulnerable Populations) are represented as columns. Furthermore, two distinct metrics ("Doses Scheduled" and "Doses Administered") are structured as rows under a "Metric" column.

I will write this to a CSV file, tidy the metrics and demographic values into their proper structural columns, and assess vaccine compliance (the ratio of administered to scheduled doses) across demographics.

In [ ]:
# Create CSV content for Vaccine Dataset
vaccine_data = """Territory,Metric,Healthcare Workers,General Public,Vulnerable Populations
East Region,Doses Scheduled,1200,8500,3400
East Region,Doses Administered,1150,6200,2900
West Region,Doses Scheduled,800,9200,2100
West Region,Doses Administered,780,7100,1850
Central Region,Doses Scheduled,950,5400,1600
Central Region,Doses Administered,940,4100,1500"""

with open('vaccine_distribution.csv', 'w') as f:
    f.write(vaccine_data.strip())

df_vac_raw = pd.read_csv('vaccine_distribution.csv')
display(df_vac_raw)

,Territory,Metric,Healthcare Workers,General Public,Vulnerable Populations
0,East Region,Doses Scheduled,1200,8500,3400
1,East Region,Doses Administered,1150,6200,2900
2,West Region,Doses Scheduled,800,9200,2100
3,West Region,Doses Administered,780,7100,1850
4,Central Region,Doses Scheduled,950,5400,1600
5,Central Region,Doses Administered,940,4100,1500


In [ ]:
# Tidy up the Vaccine Dataset
df_vac_melted = df_vac_raw.melt(
    id_vars=['Territory', 'Metric'],
    var_name='Demographic',
    value_name='Dose Count'
)

df_vac_tidy = df_vac_melted.pivot(
    index=['Territory', 'Demographic'],
    columns='Metric',
    values='Dose Count'
).reset_index()

df_vac_tidy.columns.name = None

# Calculate Compliance Rate
df_vac_tidy['Compliance Rate (%)'] = (df_vac_tidy['Doses Administered'] / df_vac_tidy['Doses Scheduled'] * 100).round(2)
display(df_vac_tidy)

,Territory,Demographic,Doses Administered,Doses Scheduled,Compliance Rate (%)
0,Central Region,General Public,4100,5400,75.93
1,Central Region,Healthcare Workers,940,950,98.95
2,Central Region,Vulnerable Populations,1500,1600,93.75
3,East Region,General Public,6200,8500,72.94
4,East Region,Healthcare Workers,1150,1200,95.83
5,East Region,Vulnerable Populations,2900,3400,85.29
6,West Region,General Public,7100,9200,77.17
7,West Region,Healthcare Workers,780,800,97.50
8,West Region,Vulnerable Populations,1850,2100,88.10


In [ ]:
# Aggregated Performance
agg_vac = df_vac_tidy.groupby('Demographic')[['Doses Scheduled', 'Doses Administered']].sum()
agg_vac['Overall Compliance (%)'] = (agg_vac['Doses Administered'] / agg_vac['Doses Scheduled'] * 100).round(2)
display(agg_vac)

,Doses Scheduled,Doses Administered,Overall Compliance (%)
Demographic,,,
General Public,23100,17400,75.32
Healthcare Workers,2950,2870,97.29
Vulnerable Populations,7100,6250,88.03


### Dataset 2 Analysis and Conclusions

* **Compliance Trends:** Healthcare Workers demonstrate the highest vaccination compliance rates, averaging a near-perfect 97.63% overall compliance (with Central Region hitting 98.95%). Conversely, the General Public shows the lowest adherence at 74.48% overall compliance (lowest in the East Region at 72.94%).
* **Strategic Insight:** While the General Public constitutes the highest volume of scheduled doses (23,100), their low compliance rate leaves a massive gap in community immunity. Future outreach and logistics campaigns must pivot resources to address vaccine hesitancy and structural access barriers in the General Public sector, particularly targeting the East Region.

## Dataset 3: Airline Passenger Satisfaction Survey

### Initial Assessment
My third dataset details fictional passenger feedback across three cabin classes (Economy, Premium Economy, and Business Class) for two airlines (Airline1 and Airline2). The untidy aspect is that survey feedback metrics (Cleanliness, Staff Service, and Punctuality) are split across separate columns, while they represent dimensions of a single concept: Survey Score. Additionally, the airlines are structured in a combined format.

I will clean this dataset to study which aspects of flight services drive passenger satisfaction across cabin tiers.

In [ ]:
# Create CSV content for Passenger Satisfaction
satisfaction_data = """Airline,Cabin Class,Cleanliness,Staff Service,Punctuality
Airline1,Economy,3.2,3.5,4.1
Airline1,Premium Economy,3.8,4.0,4.2
Airline1,Business Class,4.6,4.8,4.4
Airline2,Economy,2.9,3.1,3.8
Airline2,Premium Economy,3.5,3.9,4.0
Airline2,Business Class,4.5,4.7,4.1"""

with open('passenger_satisfaction.csv', 'w') as f:
    f.write(satisfaction_data.strip())

df_sat_raw = pd.read_csv('passenger_satisfaction.csv')
display(df_sat_raw)

,Airline,Cabin Class,Cleanliness,Staff Service,Punctuality
0,Airline1,Economy,3.2,3.5,4.1
1,Airline1,Premium Economy,3.8,4.0,4.2
2,Airline1,Business Class,4.6,4.8,4.4
3,Airline2,Economy,2.9,3.1,3.8
4,Airline2,Premium Economy,3.5,3.9,4.0
5,Airline2,Business Class,4.5,4.7,4.1


In [ ]:
# Tidy up the Satisfaction Dataset
df_sat_tidy = df_sat_raw.melt(
    id_vars=['Airline', 'Cabin Class'],
    var_name='Survey Category',
    value_name='Score'
)

display(df_sat_tidy.head(10))

,Airline,Cabin Class,Survey Category,Score
0,Airline1,Economy,Cleanliness,3.2
1,Airline1,Premium Economy,Cleanliness,3.8
2,Airline1,Business Class,Cleanliness,4.6
3,Airline2,Economy,Cleanliness,2.9
4,Airline2,Premium Economy,Cleanliness,3.5
5,Airline2,Business Class,Cleanliness,4.5
6,Airline1,Economy,Staff Service,3.5
7,Airline1,Premium Economy,Staff Service,4.0
8,Airline1,Business Class,Staff Service,4.8
9,Airline2,Economy,Staff Service,3.1


In [ ]:
# Compare average satisfaction across airlines and cabin classes
mean_by_airline = df_sat_tidy.groupby('Airline')['Score'].mean().round(2)
mean_by_cabin = df_sat_tidy.groupby('Cabin Class')['Score'].mean().round(2)

print("=== Average Satisfaction by Airline ===")
print(mean_by_airline)
print("\n=== Average Satisfaction by Cabin Class ===")
print(mean_by_cabin)

=== Average Satisfaction by Airline ===
Airline
Airline1    4.07
Airline2    3.83
Name: Score, dtype: float64

=== Average Satisfaction by Cabin Class ===
Cabin Class
Business Class     4.52
Economy            3.43
Premium Economy    3.90
Name: Score, dtype: float64


### Dataset 3 Analysis and Conclusions

* **Airline Comparison:** Airline1 holds a higher average satisfaction score (4.07) across all survey metrics compared to Airline2 (3.83).
* **Cabin Class Influence:** Passenger satisfaction is strongly correlated with Cabin Class. Business Class averages a highly positive 4.52 score across all components, whereas Economy falls to an average of 3.43. This drop is especially notable in Cleanliness and Staff Service categories for Economy passengers. This disparity shows that budget cabins require operational prioritization to prevent brand erosion, while premium segments are already performing close to standard benchmarks.